# 🏗️ EmPay Project Architecture

This notebook contains visual diagrams of the application architecture.

### 1. High-Level System Architecture
This diagram outlines the technical stack and how data flows from the React frontend, through the Express backend, and into the PostgreSQL database.

```mermaid
graph TD
    %% Define Modern Hex Colors
    classDef frontend fill:#2563eb,stroke:#1e40af,stroke-width:2px,color:#ffffff
    classDef backend fill:#059669,stroke:#047857,stroke-width:2px,color:#ffffff
    classDef database fill:#ea580c,stroke:#c2410c,stroke-width:2px,color:#ffffff
    classDef auth fill:#7c3aed,stroke:#6d28d9,stroke-width:2px,color:#ffffff

    subgraph "Frontend (Client)"
        UI[React 19 + Vite UI]:::frontend
        Tailwind[Tailwind CSS v4 + shadcn/ui]:::frontend
        Axios[Axios API Client]:::frontend
        
        UI --> Tailwind
        UI --> Axios
    end

    subgraph "Backend (Server)"
        Express[Node.js + Express.js Router]:::backend
        JWT[JWT Auth Middleware]:::auth
        
        subgraph "Controllers (Business Logic)"
            UserCtrl[User & Role Controller]:::backend
            AttCtrl[Attendance Controller]:::backend
            LeaveCtrl[Leave Controller]:::backend
            PayCtrl[Payroll Controller]:::backend
        end
        
        Prisma[Prisma ORM]:::backend
        
        Express -->|Intercepts Request| JWT
        JWT -->|Validates & Routes| UserCtrl
        JWT -->|Validates & Routes| AttCtrl
        JWT -->|Validates & Routes| LeaveCtrl
        JWT -->|Validates & Routes| PayCtrl
        
        UserCtrl --> Prisma
        AttCtrl --> Prisma
        LeaveCtrl --> Prisma
        PayCtrl --> Prisma
    end

    subgraph "Database Layer"
        PostgreSQL[(PostgreSQL)]:::database
        Users[(Users Table)]:::database
        Attendance[(Attendance Table)]:::database
        Leaves[(Leaves Table)]:::database
        Payrolls[(Payroll Table)]:::database
        
        PostgreSQL --- Users
        PostgreSQL --- Attendance
        PostgreSQL --- Leaves
        PostgreSQL --- Payrolls
    end

    %% Flow Connections
    Axios -- "HTTP REST Requests" --> Express
    Axios -. "Attaches Bearer Token" .-> JWT
    Prisma -- "SQL Queries" --> PostgreSQL
```


---

### 2. Business Logic & Data Flow (The Payroll Pipeline)
This flowchart shows exactly how your four user roles interact with the system modules, highlighting how employee actions directly drive the final payroll generation.

```mermaid
flowchart LR
    %% Define Role Colors
    classDef admin fill:#dc2626,color:#fff,stroke:#b91c1c,stroke-width:2px
    classDef hr fill:#d97706,color:#fff,stroke:#b45309,stroke-width:2px
    classDef payroll fill:#0284c7,color:#fff,stroke:#0369a1,stroke-width:2px
    classDef employee fill:#4f46e5,color:#fff,stroke:#4338ca,stroke-width:2px
    classDef module fill:#f3f4f6,color:#111827,stroke:#9ca3af,stroke-width:2px

    %% Users
    Admin((Admin)):::admin
    HROfficer((HR Officer)):::hr
    PayrollOfficer((Payroll Officer)):::payroll
    Employee((Employee)):::employee

    %% Admin Flow
    Admin -->|Creates Accounts| HROfficer
    Admin -->|Creates Accounts| PayrollOfficer
    Admin -->|Full Access| SystemModules[All System Modules]:::module

    %% HR Flow
    HROfficer -->|Creates Profiles| EmpProfile[Employee Profiles]:::module
    HROfficer -->|Allocates| LeaveBalances[Leave Balances]:::module
    HROfficer -->|Monitors| AttRecords[Attendance Records]:::module

    %% Employee Flow
    Employee -->|Marks Daily| AttRecords
    Employee -->|Applies For| LeaveReq[Leave Requests]:::module

    %% Payroll Pipeline Flow
    PayrollOfficer -->|Approves/Rejects| LeaveReq
    
    AttRecords -.->|Feeds Data Into| PayrollPipeline[Payroll Module]:::module
    LeaveReq -.->|Feeds Data Into| PayrollPipeline
    EmpProfile -.->|Base Salary Info| PayrollPipeline
    
    PayrollOfficer -->|Processes| PayrollPipeline
    
    PayrollPipeline -->|Calculates Basic Pay| Calculations{Calculations}
    Calculations -->|Deducts 12% PF| Deductions
    Calculations -->|Deducts Prof. Tax| Deductions
    
    Deductions -->|Generates| Payslip[Final Payslip]:::module
```
